<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 5 · Reading a response</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">Reading a response</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Stop reasons, usage, the errors a call can return, and retries that back off, measured on recorded responses and on failures simulated in the notebook.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

A response says why it stopped, what it cost and, when the call fails, what kind of failure it was. This notebook reads all three on Claude Sonnet 5.5. It provokes each stop reason a plain request can produce, prints every usage field, sends four requests that fail in different ways, then retries through failures the API returns when it is busy, which the notebook simulates because a recording can't summon them. The retrying comes from `retry.py`, which Claude Code generated from the chapter's spec and tests.

## Setup

This cell installs Anthropic's Python library and builds the client every call below goes through. It replays recorded answers unless you set `LIVE = True`.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/reading-a-response.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("reading-a-response", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 9 recorded responses from claude-sonnet-5-5, claude-sonnet-9-9, recorded 2026-11-12. Set LIVE = True in Setup to call the API yourself.


## The module from the chapter's Claude Code build

The notebook's retries run through `retry.py`, which Claude Code generated in the chapter's build from a spec and 29 tests. The file lives in the book's repository, in `builds/reading-a-response/generated/`, next to the spec, the tests and a transcript of what Claude Code did. This cell loads it and prints its source, so `is_retryable`, `retry_after`, `backoff_delay`, `call_with_retries` and `create_with_retries` are on the page before the notebook calls them. In Colab the cell downloads the file next to the notebook, and the Files panel on the left shows it.

In [2]:
# retry.py is the module Claude Code generated in this chapter's build, from the spec and tests in
# builds/reading-a-response/. Colab fetches it from the book's repository.
BUILD = "reading-a-response"
if "google.colab" in sys.modules:
    urllib.request.urlretrieve("https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/builds/"
                               f"{BUILD}/generated/retry.py", "retry.py")
else:
    sys.path.insert(0, str(pathlib.Path("../../builds", BUILD, "generated").resolve()))
import retry
retry = importlib.reload(retry)
from retry import backoff_delay, call_with_retries, create_with_retries, is_retryable, retry_after

# The whole module, so you can read every function before the notebook uses it
import inspect
print(inspect.getsource(retry))

import math
import random
import time

import anthropic


def _spend_cap_reached(error):
    body = getattr(error, "body", None)
    if not isinstance(body, dict):
        return False
    inner = body.get("error")
    if not isinstance(inner, dict):
        return False
    details = inner.get("details")
    return isinstance(details, dict) and details.get("error_code") == "enforced_spend_limit_reached"


def is_retryable(error):
    if isinstance(error, anthropic.APIConnectionError):
        return True
    if isinstance(error, anthropic.APIStatusError):
        status = error.status_code
        if status == 429 and _spend_cap_reached(error):
            return False
        return status in (408, 409, 429) or status >= 500
    return False


def retry_after(error):
    response = getattr(error, "response", None)
    if response is None:
        return None
    value = response.headers.get("retry-after")
    if value is None:
        return None
    try:
        seconds = float(valu

## Why a reply stopped

`stop_reason` says why generation ended. The cell provokes the three a plain request can produce: `end_turn`, when the model finishes, `max_tokens`, when the reply reaches its cap, and `stop_sequence`, when the reply produces a string the request named. It also sends the capped question again with room to finish.

In [3]:
MODEL = "claude-sonnet-5-5"


def answer_text(reply):
    """The text of a reply: every text block, joined."""
    return "".join(block.text for block in reply.content if block.type == "text")


def describe(name, reply):
    u = reply.usage
    print(f"== {name}")
    print(f"stop_reason {reply.stop_reason!r}, stop_sequence {reply.stop_sequence!r}, "
          f"blocks {[b.type for b in reply.content]}")
    print(f"{u.input_tokens} tokens in, {u.output_tokens} out, {u.output_tokens_details.thinking_tokens} of them thinking")
    print(repr(answer_text(reply)), "\n")


QUESTION = "Explain how a support ticket triage system works."
describe("a short answer", client.messages.create(
    model=MODEL, max_tokens=1024, messages=[{"role": "user", "content": "Name the capital of France in one word."}]))
describe("max_tokens=40", client.messages.create(
    model=MODEL, max_tokens=40, messages=[{"role": "user", "content": QUESTION}]))
full = client.messages.create(model=MODEL, max_tokens=4096, messages=[{"role": "user", "content": QUESTION}])
print(f"== the same question with max_tokens=4096\nstop_reason {full.stop_reason!r}, "
      f"{full.usage.output_tokens} tokens out, {full.usage.output_tokens_details.thinking_tokens} of them thinking, "
      f"{len(answer_text(full))} characters of text\n")
describe("stop_sequences=['4.']", client.messages.create(
    model=MODEL, max_tokens=1024, stop_sequences=["4."],
    messages=[{"role": "user", "content": "List six steps for handling a damaged-item complaint, numbered 1. to 6., "
                                          "one line each."}]))

== a short answer
stop_reason 'end_turn', stop_sequence None, blocks ['text']
19 tokens in, 5 out, 0 of them thinking
'Paris' 

== max_tokens=40
stop_reason 'max_tokens', stop_sequence None, blocks ['thinking']
22 tokens in, 40 out, 40 of them thinking
'' 

== the same question with max_tokens=4096
stop_reason 'end_turn', 2150 tokens out, 109 of them thinking, 5545 characters of text

== stop_sequences=['4.']
stop_reason 'stop_sequence', stop_sequence '4.', blocks ['text']
39 tokens in, 123 out, 0 of them thinking
'1. Acknowledge the complaint promptly, apologize, and thank the customer for reporting it.\n2. Collect details: order number, photos of the damage, and packaging condition.\n3. Verify the claim against the order record, shipping data, and return policy.\n' 



The one-word question ended with `end_turn` after 5 output tokens. With `max_tokens=40` the explanation stopped at `max_tokens`, and all 40 tokens were thinking, so the reply held a thinking block and no text at all. The same question with room to finish took 2,150 output tokens, 109 of them thinking, and 5,545 characters of text, which shows how far short the cap was. The numbered list stopped at the stop sequence `4.` after three items, and the returned text leaves the stop sequence out. Code that reads `answer_text` without checking `stop_reason` would have treated the empty string as a complete answer.

## Every usage field

`usage` reports what the call consumed. The cell prints all of it for the full-length answer.

In [4]:
import json

# Every field the full answer's usage reports
print(json.dumps(full.usage.to_dict(), indent=2))

{
  "cache_creation": {
    "ephemeral_1h_input_tokens": 0,
    "ephemeral_5m_input_tokens": 0
  },
  "cache_creation_input_tokens": 0,
  "cache_read_input_tokens": 0,
  "inference_geo": "global",
  "input_tokens": 22,
  "output_tokens": 2150,
  "output_tokens_details": {
    "thinking_tokens": 109
  },
  "service_tier": "standard",
  "speed": "standard"
}


`input_tokens` and `output_tokens` are the counts Chapter 3 priced, and `output_tokens_details.thinking_tokens` says how many of the output tokens were thinking, 109 of 2,150 here. The two cache counts and `cache_creation`'s breakdown by lifetime are 0 because the request used no prompt caching, which Chapter 118, Caching and batching, turns on. `service_tier` and `speed` read `standard`, and `inference_geo`, which reports where the request ran, reads `global`.

## Four ways a request fails

A failed call raises an SDK exception whose class matches the HTTP status, with the API's error `type` and message in its `body`. The cell sends three requests the API rejects and one the SDK refuses before sending, and asks `is_retryable` about each.

In [5]:
import anthropic

BAD_REQUESTS = {
    "a model that doesn't exist": dict(model="claude-sonnet-9-9", max_tokens=100,
                                       messages=[{"role": "user", "content": "Hello."}]),
    "no messages": dict(model=MODEL, max_tokens=100, messages=[]),
    "an empty message": dict(model=MODEL, max_tokens=100, messages=[{"role": "user", "content": ""}]),
    "max_tokens=200000": dict(model=MODEL, max_tokens=200_000, messages=[{"role": "user", "content": "Hello."}]),
}

for name, request in BAD_REQUESTS.items():
    print(f"== {name}")
    try:
        client.messages.create(**request)
    except anthropic.APIStatusError as e:
        print(f"{type(e).__name__}, status {e.status_code}, type {e.body['error']['type']!r}, "
              f"retryable: {is_retryable(e)}")
        print(e.body["error"]["message"], "\n")
    except ValueError as e:
        print(f"ValueError raised by the SDK before sending, retryable: {is_retryable(e)}")
        print(str(e), "\n")

== a model that doesn't exist
NotFoundError, status 404, type 'not_found_error', retryable: False
model: claude-sonnet-9-9 

== no messages
BadRequestError, status 400, type 'invalid_request_error', retryable: False
messages: at least one message is required 

== an empty message
BadRequestError, status 400, type 'invalid_request_error', retryable: False
messages.0: user messages must have non-empty content 

== max_tokens=200000
ValueError raised by the SDK before sending, retryable: False
Streaming is required for operations that may take longer than 10 minutes. See https://platform.claude.com/docs/en/cli-sdks-libraries/sdks/python#long-requests for more details 



The unknown model raised `NotFoundError` with status 404 and the type `not_found_error`. The empty message list and the empty message each raised `BadRequestError`, status 400, `invalid_request_error`, with messages that name the field at fault. `max_tokens=200000` never reached the API. The SDK raised a `ValueError` because a request that large needs streaming, as of SDK 2.7.0. None of the four is retryable, since sending the same request again fails the same way.

## Retrying through a busy API

Rate limits (429) and overload (529) depend on traffic, so a recording can't produce them on demand. The cell builds the two errors the way the SDK does, from responses in the shape of Anthropic's error bodies that say they are simulated, raises them in front of a real request, and lets `call_with_retries` work through them. The sleep is replaced by a list, so the notebook records each wait without waiting.

In [6]:
import random
import httpx2

# Two failures the API returns when it is busy, written here because a recording can't summon them on demand.
# Their bodies follow the shape of Anthropic's error responses and say they are simulated.
def simulated(status, error_type, headers=None):
    request = httpx2.Request("POST", "https://api.anthropic.com/v1/messages")
    body = {"type": "error", "error": {"type": error_type, "message": "Simulated for the notebook."}}
    response = httpx2.Response(status, request=request, headers=headers or {}, json=body)
    cls = {429: anthropic.RateLimitError, 529: anthropic.OverloadedError}[status]
    return cls("Simulated for the notebook.", response=response, body=body)


failures = [simulated(529, "overloaded_error"), simulated(429, "rate_limit_error", {"retry-after": "2"})]


def send():
    """The real request, preceded by the two simulated failures."""
    if failures:
        raise failures.pop(0)
    return client.messages.create(model=MODEL, max_tokens=1024,
                                  messages=[{"role": "user", "content": "Name the capital of France in one word."}])


waits = []
reply = call_with_retries(send, sleep=waits.append, rng=random.Random(5).random,
                          on_retry=lambda n, e, d: print(f"attempt {n} failed with {type(e).__name__} "
                                                         f"({e.status_code}), waiting {d:.2f} s"))
print(f"attempt 3 answered {answer_text(reply)!r}, after waits of {[round(w, 2) for w in waits]} seconds")

# A request error is raised at once: retrying the same bad request would fail the same way
tries = []
try:
    call_with_retries(lambda: tries.append(1) or client.messages.create(**BAD_REQUESTS["a model that doesn't exist"]),
                      sleep=waits.append)
except anthropic.NotFoundError as e:
    print(f"\nNotFoundError after {len(tries)} attempt, no wait")

attempt 1 failed with OverloadedError (529), waiting 0.62 s
attempt 2 failed with RateLimitError (429), waiting 2.00 s
attempt 3 answered 'Paris', after waits of [0.62, 2.0] seconds

NotFoundError after 1 attempt, no wait


The simulated 529 carried no `retry-after` header, so the first wait came from exponential backoff with jitter, 0.62 seconds. The simulated 429 said `retry-after: 2`, and the second wait was exactly 2 seconds. The third attempt reached the recorded answer, Paris. The request for a model that doesn't exist was raised after one attempt with no wait, because `is_retryable` puts a 404 among the errors no retry can fix.

## The SDK's own retries

Anthropic's SDK retries by itself. The cell builds a client with the default settings, sends nothing with it, and compares its settings with the notebook's replay client.

In [7]:
# The SDK retries on its own. A client built with defaults (no request is sent here) shows its settings.
default = anthropic.Anthropic(api_key="not-used-no-request-is-sent")
print(f"a default client: max_retries {default.max_retries}, read timeout {default.timeout.read} seconds")
print("the notebook's replay client: max_retries", client.max_retries)

# Retries stacked on retries multiply the attempts on every failure that is retryable
sdk_attempts, wrapper_attempts = default.max_retries + 1, 4
print(f"SDK inside the wrapper: up to {sdk_attempts} x {wrapper_attempts} = {sdk_attempts * wrapper_attempts} "
      "requests for one call")

a default client: max_retries 2, read timeout 600 seconds
the notebook's replay client: max_retries 0
SDK inside the wrapper: up to 3 x 4 = 12 requests for one call


A default client retries twice, so it makes up to three attempts, and waits up to 600 seconds to read a response, as of SDK 2.7.0. The replay client is built with `max_retries` set to 0, since a recorded answer never needs a retry. Wrapping a default client in `call_with_retries` would multiply the attempts, up to 3 times 4, or 12 requests for one call, which is why `create_with_retries` turns the SDK's retries off and keeps one place in charge of waiting.

## What the notebook showed

- `stop_reason` separates a finished reply from a cut-off one. With `max_tokens=40` the reply was all thinking and held no text.
- A stop sequence ends the reply where the request said, and the text leaves it out.
- `usage` splits the input by cache use and the output by thinking.
- Request errors (400 and 404) arrive as SDK exceptions with the API's type and message, and a retry can't fix them.
- A 529 is retried after a jittered backoff, and a 429 after the wait its `retry-after` header names.
- The SDK retries twice by default, so a wrapper of its own should turn that off.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>